In [0]:
%pip install --upgrade mlflow --pre openai
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient

import mlflow
from mlflow.metrics.genai import answer_correctness, answer_similarity, faithfulness

# turn on autologging for automatic tracing
mlflow.openai.autolog()

# Initialize Workspace Client
client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# define a prompt template
prompt_template = """\
You are an expert AI assistant. Answer the user's question with clarity, accuracy, and conciseness.

## Question:
{question}

## Guidelines:
- Keep responses factual and to the point.
- If relevant, provide examples or step-by-step instructions.
- If the question is ambiguous, clarify before answering.

Respond below:
"""

# groundtruth result for evaluation
mlflow_ground_truth = (
    "MLflow is an open-source platform for managing "
    "the end-to-end machine learning (ML) lifecycle. It was developed by Databricks, "
    "a company that specializes in big data and machine learning solutions. MLflow is "
    "designed to address the challenges that data scientists and machine learning "
    "engineers face when developing, training, and deploying machine learning models."
)

databricks_model = "endpoints:/databricks-meta-llama-3-3-70b-instruct"

# Define evaluation metrics
metrics = {
    "answer_similarity": answer_similarity(model=databricks_model),
    "answer_correctness": answer_correctness(model=databricks_model),
    "faithfulness": faithfulness(model=databricks_model),
}
question = "What is MLflow?"

mlflow.set_active_model(name="My agent") # Added by Corey Zumar

# # Start a run to represent the evaluation process
with mlflow.start_run():
    response = (
        client.chat.completions.create(
            messages=[
                {"role": "user", "content": prompt_template.format(question=question)}
            ],
            model="databricks-meta-llama-3-3-70b-instruct",
            temperature=0.1,
            max_tokens=2000,
        )
        .choices[0]
        .message.content
    )

    # Calculate metrics based on the input, response and ground truth
    # The evaluation metrics are callables that can be invoked directly
    answer_similarity_score = metrics["answer_similarity"](
        predictions=response, inputs=question, targets=mlflow_ground_truth
    ).scores[0]
    answer_correctness_score = metrics["answer_correctness"](
        predictions=response, inputs=question, targets=mlflow_ground_truth
    ).scores[0]
    faithfulness_score = metrics["faithfulness"](
        predictions=response, inputs=question, context=mlflow_ground_truth
    ).scores[0]

    # Fetch the LoggedModel that's automatically created during autologging
    logged_model_id = mlflow.get_active_model_id()
    # Log metrics and pass model_id to link the metrics
    mlflow.log_metrics(
        {
            "answer_similarity": answer_similarity_score,
            "answer_correctness": answer_correctness_score,
            "faithfulness": faithfulness_score,
        },
        model_id=logged_model_id
    )

print(f"LoggedModel model id: {logged_model_id}")

traces = mlflow.search_traces(model_id=logged_model_id)
print(traces)